# Metop (EPS) / IASI (Infrared Atmospheric Sounding Interferometer)

Example notebook from the [Satellite Data Catalog](https://alessandrosebastianelli.github.io/satcat/satellite.html?folder=metop/metop-iasi). Each section below runs the Python example of one processing level.

Run the cells top to bottom. Some sources need a free account; the cell that needs credentials asks for them when it runs.

## L1C

Level 1C: apodised, geolocated and calibrated spectra, with AVHRR-based cloud information, per field of view (2x2 pixel of 12 km in 50 km footprints).

Downloads one IASI L1C product from the EUMETSAT Data Store (free account and consumer key/secret, asked at run time) and opens it.

In [ ]:
%pip install -q eumdac satpy netCDF4 h5netcdf matplotlib

In [ ]:
import datetime as dt
import getpass
import shutil
import zipfile
from pathlib import Path

import eumdac

# Free consumer key / secret: https://api.eumetsat.int/api-key (EUMETSAT Data Store account)
token = eumdac.AccessToken((input("EUMETSAT consumer key: "), getpass.getpass("EUMETSAT consumer secret: ")))
datastore = eumdac.DataStore(token)
collection = datastore.get_collection("EO:EUM:DAT:METOP:IASIL1C-ALL")

products = list(collection.search(dtstart=dt.datetime(2024, 6, 1, 12, 0), dtend=dt.datetime(2024, 6, 1, 12, 20)))
print(len(products), "IASI L1C products found")
if not products:
    raise SystemExit("Nothing in this time window - try another one")
product = products[0]
print(product)

out = Path("data")
out.mkdir(exist_ok=True)
# a few hundred MB per product
with product.open() as src, open(out / src.name, "wb") as dst:
    shutil.copyfileobj(src, dst)
for z in out.glob("*.zip"):                     # some collections arrive zipped
    with zipfile.ZipFile(z) as zf:
        zf.extractall(out / z.stem)

import xarray as xr

files = sorted(out.rglob("*.nc"))
print(files[:3])
if files:
    try:
        ds = xr.open_dataset(files[0], engine="h5netcdf")
        print(ds)
    except Exception as exc:                      # some products need group="..." or another engine
        print("Could not open directly:", exc)
else:
    print("No .nc file: the product is in EPS native / BUFR / HDF5 format - read it with satpy, eccodes or the EUMETSAT tools")

## L2

Level 2: retrieved profiles of temperature and water vapour, surface temperature and emissivity, cloud parameters, O3, CO and other trace gases (operational IASI L2 PPF products).

Downloads one IASI L2 sounding product from the EUMETSAT Data Store (free account and consumer key/secret, asked at run time) and opens it.

In [ ]:
%pip install -q eumdac satpy netCDF4 h5netcdf matplotlib

In [ ]:
import datetime as dt
import getpass
import shutil
import zipfile
from pathlib import Path

import eumdac

# Free consumer key / secret: https://api.eumetsat.int/api-key (EUMETSAT Data Store account)
token = eumdac.AccessToken((input("EUMETSAT consumer key: "), getpass.getpass("EUMETSAT consumer secret: ")))
datastore = eumdac.DataStore(token)
collection = datastore.get_collection("EO:EUM:DAT:METOP:IASSND02")

products = list(collection.search(dtstart=dt.datetime(2024, 6, 1, 12, 0), dtend=dt.datetime(2024, 6, 1, 12, 20)))
print(len(products), "IASI L2 sounding products found")
if not products:
    raise SystemExit("Nothing in this time window - try another one")
product = products[0]
print(product)

out = Path("data")
out.mkdir(exist_ok=True)
# tens of MB per product
with product.open() as src, open(out / src.name, "wb") as dst:
    shutil.copyfileobj(src, dst)
for z in out.glob("*.zip"):                     # some collections arrive zipped
    with zipfile.ZipFile(z) as zf:
        zf.extractall(out / z.stem)

import xarray as xr

files = sorted(out.rglob("*.nc"))
print(files[:3])
if files:
    try:
        ds = xr.open_dataset(files[0], engine="h5netcdf")
        print(ds)
    except Exception as exc:                      # some products need group="..." or another engine
        print("Could not open directly:", exc)
else:
    print("No .nc file: the product is in EPS native / BUFR / HDF5 format - read it with satpy, eccodes or the EUMETSAT tools")